In [ ]:
import time, torch, platform
print ("System: ", platform.platform())
print("Torch :", torch.__version__, "| CUDA available:", torch.cuda.is_available())


System:  Linux-6.6.122+-x86_64-with-glibc2.35
Torch : 2.11.0+cu128 | CUDA available: True


In [ ]:
#cpu benchmark
size =4096
a_cpu = torch.randn(size,size)
b_cpu = torch.randn (size, size)

t0 = time.perf_counter() #start counter
c_cpu = a_cpu @ b_cpu #multiple both matrices
t_cpu = time.perf_counter() - t0

print (f"CPU matmul {size}x{size}:{t_cpu*1000:.1f}ms")

CPU matmul 4096x4096:1106.9ms


In [ ]:
if torch.cuda.is_available():
    #Those matrices live in normal RAM and are handled by the CPU, .cuda() copies them to the GPU memory
    a_gpu = a_cpu.cuda()
    b_gpu = b_cpu.cuda()

    #Wait until the GPU finishes all previous work before continuing
    #This first synchronize makes sure the data transfer/setup is finished before starting the timer
    torch.cuda.synchronize()
    t0 = time.perf_counter()

    c_gpu = a_gpu @ b_gpu

    #This second synchronize means: Wait until the matrix multiplication is fully finished
    #Without this, Python might stop the timer too early and give a fake fast result
    torch.cuda.synchronize()
    t_gpu = time.perf_counter() - t0

    print(f"GPU matmul {size}x{size}: {t_gpu*1000:.1f} ms")
    print(f"Speedup: {t_cpu / t_gpu:.1f}x") #This compares CPU time to GPU time.
else:
    print("Skipping GPU benchmark because CUDA is not available on this machine.")
    print("This is expected on my laptop. GPU testing will be done in Google Colab.")

GPU matmul 4096x4096: 157.0 ms
Speedup: 7.1x
